# BPE Tokenizer From Scratch

This notebook implements Byte Pair Encoding (BPE) from first principles. 

### The Goal
To build a tokenizer that can:
1. **Train**: Learn merge rules from a corpus.
2. **Encode**: Turn a string into a sequence of token IDs.
3. **Decode**: Turn token IDs back into a string.

Following the "Residual Stream" philosophy: **No shortcuts, no libraries, just logic.**

In [ ]:
import jax.numpy as jnp
from typing import List, Dict, Tuple

## 1. The Core Mechanics

BPE works by finding the most frequent pair of adjacent tokens and merging them into a single new token. We repeat this until we reach our target vocabulary size.

In [ ]:
def get_stats(ids: List[int]) -> Dict[Tuple[int, int], int]:
    """Counts the frequency of all adjacent pairs in a list of integers."""
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids: List[int], pair: Tuple[int, int], new_id: int) -> List[int]:
    """Replaces all occurrences of 'pair' with 'new_id'."""
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            new_ids.append(new_id)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

## 2. The Tokenizer Class

We start with the base vocabulary of 256 (all possible byte values). Every merge rule learned increases our vocab size by one.

In [ ]:
class BPETokenizer:
    def __init__(self):
        self.merges = {}  # (int, int) -> int
        self.vocab = {i: bytes([i]) for i in range(256)} # int -> bytes
    
    def train(self, text: str, vocab_size: int, verbose: bool = True):
        num_merges = vocab_size - 256
        # Convert text to raw bytes, then to integers
        tokens = list(text.encode("utf-8"))
        
        for i in range(num_merges):
            stats = get_stats(tokens)
            if not stats:
                break
            
            # Find the most frequent pair
            best_pair = max(stats, key=stats.get)
            new_id = 256 + i
            
            # Record the merge
            self.merges[best_pair] = new_id
            self.vocab[new_id] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]
            
            # Update token list for next iteration
            tokens = merge(tokens, best_pair, new_id)
            
            if verbose and (i + 1) % 5 == 0:
                print(f"Merge {i+1}/{num_merges}: {best_pair} -> {new_id} ({self.vocab[new_id].decode('utf-8', errors='replace')})")

    def encode(self, text: str) -> List[int]:
        tokens = list(text.encode("utf-8"))
        # We must apply merges in the same order they were learned
        while len(tokens) >= 2:
            stats = get_stats(tokens)
            # Look for the earliest merge rule that exists in our current tokens
            pair = min(stats.keys(), key=lambda p: self.merges.get(p, float('inf')))
            
            if pair not in self.merges:
                break # No more merges to apply
                
            tokens = merge(tokens, pair, self.merges[pair])
        return tokens

    def decode(self, ids: List[int]) -> str:
        # Concatenate bytes from vocab and decode back to string
        text_bytes = b"".join(self.vocab[idx] for idx in ids)
        return text_bytes.decode("utf-8", errors="replace")

## 3. Testing on a Toy Corpus

Let's see if it learns common patterns like "the" or "ing".

In [ ]:
tokenizer = BPETokenizer()
corpus = """
The residual stream is where the information flows.
In the transformer, the residual stream keeps the gradients healthy.
We are building the residual stream from scratch.
"""

print("Training...")
tokenizer.train(corpus, vocab_size=275)

test_text = "the residual stream flows"
encoded = tokenizer.encode(test_text)
decoded = tokenizer.decode(encoded)

print(f"\nOriginal: {test_text}")
print(f"Encoded:  {encoded}")
print(f"Decoded:  {decoded}")
assert test_text == decoded, "Round-trip failed!"

## Observations for the Learning Log

1. **Greedy Nature**: BPE is greedy. It picks the most frequent pair globally, which might not be optimal for every single sentence.
2. **UTF-8 Handling**: By starting with bytes (0-255), we never encounter an "out-of-vocabulary" (OOV) error. Every character can be broken down into bytes if it hasn't been merged.
3. **Merge Order**: During encoding, it is critical to apply merges in the order they were learned (the `min` key in `encode`).